# 01 — Data Preparation

Load full BIRD dev, optionally inspect Spider, and create SFT subsets only when an explicit synthetic-training JSONL is supplied.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def bootstrap_repo():
    configured = os.environ.get('SLM_PRJ_PATH')
    candidates = [Path(configured).expanduser()] if configured else []
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents,
                       Path('/kaggle/working/SLM-PRJ'),
                       Path('/kaggle/working/SLMSQL_project')])
    for candidate in candidates:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'benchmarks').is_dir():
            return candidate
    target = Path('/kaggle/working/SLM-PRJ')
    url = os.environ.get('SLM_PRJ_REPO_URL', 'https://github.com/minhadcaz/SLMSQL_project.git')
    subprocess.run(['git', 'clone', url, str(target)], check=True)
    return target

REPO_DIR = bootstrap_repo()
sys.path.insert(0, str(REPO_DIR))
from scripts.kaggle_utils import prepare_repo_import, resolve_data_dir
REPO_DIR = prepare_repo_import(REPO_DIR)
DATA_DIR = resolve_data_dir(REPO_DIR)

BIRD_DIR = DATA_DIR / 'bird'
SPIDER_DIR = DATA_DIR / 'spider'
from scripts.kaggle_utils import resolve_synsql_raw
SYNSQL_RAW = resolve_synsql_raw(REPO_DIR)
OUTPUT_DIR = DATA_DIR / 'sft_subsets'

print(f'Repo: {REPO_DIR}')
print(f'Data: {DATA_DIR}')

## 1. Load BIRD dev

In [ ]:
from benchmarks.bird_loader import load_bird_dev, get_db_path as bird_db_path
from collections import Counter

bird_dev = load_bird_dev(BIRD_DIR)
print(f'BIRD dev: {len(bird_dev)} questions')
print(f'Databases: {len(set(r["db_id"] for r in bird_dev))}')
print('Difficulty distribution:')
for k, v in sorted(Counter(r['difficulty'] for r in bird_dev).items()):
    print(f'  {k}: {v}')

## 2. Load Spider dev

In [ ]:
from benchmarks.spider_loader import load_spider_dev, get_db_path as spider_db_path

if (SPIDER_DIR / 'dev.json').is_file():
    spider_dev = load_spider_dev(SPIDER_DIR)
    print(f'Spider dev: {len(spider_dev)} questions')
    print(f'Databases: {len(set(r["db_id"] for r in spider_dev))}')
else:
    spider_dev = []
    print('Spider is optional and is not attached; skipping it.')


## 3. Sanity check — EX gold vs gold

In [ ]:
from benchmarks.eval_ex import execution_match
import time

# Test on first 50 BIRD dev questions
sample = bird_dev[:50]
start = time.time()
results = []
for r in sample:
    db = bird_db_path(BIRD_DIR, r['db_id'])
    matched = execution_match(r['gold_sql'], r['gold_sql'], db)
    results.append(matched)

elapsed = time.time() - start
acc = sum(results) / len(results)
print(f'Gold vs gold (first 50): {acc:.2%} ({sum(results)}/{len(results)})')
print(f'Time: {elapsed:.1f}s ({elapsed/len(results):.2f}s/query)')
assert acc == 1.0, f'Gold vs gold should be 100%, got {acc:.2%}'

## 4. Filter SynSQL and produce SFT subsets

In [ ]:
import json
import logging

logging.basicConfig(level=logging.INFO)

if SYNSQL_RAW is None:
    print('No SynSQL/Syn Think JSONL supplied; skipping SFT subset creation.')
    print('To enable it, attach a Kaggle dataset or set SYNSQL_RAW=/path/to/raw.jsonl.')
    prep_stats = {'skipped': True, 'reason': 'missing synthetic training JSONL'}
else:
    sizes = tuple(int(x) for x in os.environ.get('SFT_SIZES', '10000,30000,50000').split(','))
    prep_stats = prepare_subsets(
        raw_path=SYNSQL_RAW,
        output_dir=OUTPUT_DIR,
        sizes=sizes,
        max_chars=int(os.environ.get('SFT_MAX_CHARS', '8192')),
        seed=int(os.environ.get('SFT_SEED', '42')),
    )
print(json.dumps(prep_stats, indent=2, ensure_ascii=False))


## 5. Verify prompt formatting

In [ ]:
from training.prompt_template import format_sft_messages
from benchmarks.bird_loader import get_schema

example = bird_dev[0]
schema = get_schema(BIRD_DIR, example['db_id'])
messages = format_sft_messages(
    question=example['question'],
    schema=schema,
    gold_sql=example['gold_sql'],
    reasoning='Let me think step by step...',
    evidence=example.get('evidence', ''),
)

for msg in messages:
    print(f'--- {msg["role"]} ---')
    print(msg['content'][:300])
    print()